In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
class environment:
  def __init__(self, grid_height, grid_width):
    """
    The map is initialized with height and width are varible of your choice
    start: List of location where you step in, you get to the corresponding location in list 'end'
           For example:
              if you step in location start[3] then you get to new location end[3] then obtain the reward value of reward[3]
    reward: List of reward value where you move from a location in 'start' list to the corresponding location in 'end' list
    """
    self.height = grid_height
    self.width = grid_width
    self.start = []
    self.end = []
    self.reward = []
    self.map = np.array([i for i in range(grid_height * grid_width)])
    self.action_space = [0,1,2,3]
    
  def get_Map(self):
    print(self.map.reshape([self.width, self.height]))

  def get_NumState(self):
    return self.height * self.width

  def map_Designate(self, start_cell, end_cell, reward):
    self.start.append(start_cell)
    self.end.append(end_cell)
    self.reward.append(reward)
  
  def get_Observation(self, location, action):
    # If the agent at special locations, all action lead to a single location, gain reward
    if location in self.start:
      idx = self.start.index(location)
      new_location = self.end[idx]
      reward = self.reward[idx]
      return new_location, self.action_space, reward

    # If the agent not at special locations, reward = 0
    reward = 0
    new_location = 0
    # Action: UP: 0, DOWN: 1, LEFT: 2, RIGHT: 3
    # Actions that get the agent out of the map, result in no change at all
    if action == 0: #UP
      if location - self.width < 0:
        new_location = location
      else:
        new_location = location - self.width
    
    elif action == 1: #DOWN
      if location + self.width > self.height * self.width - 1:
        new_location = location
      else:
        new_location = location + self.width

    elif action == 2: #LEFT
      if location % self.width == 0:
        new_location = location 
      else:
        new_location = location - 1

    elif action == 3: #RIGHT
      if (location + 1) % self.width == 0:
        new_location = location 
      else:
        new_location = location + 1
        
    return new_location, self.action_space, reward

In [3]:
#Environment setup
Envir = environment(8,8)
Envir.get_Map()
Envir.map_Designate(17,56,-15)
Envir.map_Designate(18,56,-15)
Envir.map_Designate(19,56,-15)
Envir.map_Designate(21,56,-15)
Envir.map_Designate(25,56,-15)
Envir.map_Designate(33,56,-15)
Envir.map_Designate(41,56,-15)
Envir.map_Designate(42,56,-15)
Envir.map_Designate(43,56,-15)
Envir.map_Designate(46,56,-15)
Envir.map_Designate(47,56,-15)
Envir.map_Designate(47,56,-15)
Envir.map_Designate(15,56,+15)
Envir.map_Designate(1,10,+5)
Envir.map_Designate(26,56,+20)

# Check for the start, end, reward lists
for i in range(len(Envir.start)):
  print('i = '+ str(i) + '|Start at ' + str(Envir.start[i]) + ' results at ' + str(Envir.end[i]) + ' get Reward: ' + str(Envir.reward[i]))

[[ 0  1  2  3  4  5  6  7]
 [ 8  9 10 11 12 13 14 15]
 [16 17 18 19 20 21 22 23]
 [24 25 26 27 28 29 30 31]
 [32 33 34 35 36 37 38 39]
 [40 41 42 43 44 45 46 47]
 [48 49 50 51 52 53 54 55]
 [56 57 58 59 60 61 62 63]]
i = 0|Start at 17 results at 56 get Reward: -15
i = 1|Start at 18 results at 56 get Reward: -15
i = 2|Start at 19 results at 56 get Reward: -15
i = 3|Start at 21 results at 56 get Reward: -15
i = 4|Start at 25 results at 56 get Reward: -15
i = 5|Start at 33 results at 56 get Reward: -15
i = 6|Start at 41 results at 56 get Reward: -15
i = 7|Start at 42 results at 56 get Reward: -15
i = 8|Start at 43 results at 56 get Reward: -15
i = 9|Start at 46 results at 56 get Reward: -15
i = 10|Start at 47 results at 56 get Reward: -15
i = 11|Start at 47 results at 56 get Reward: -15
i = 12|Start at 15 results at 56 get Reward: 15
i = 13|Start at 1 results at 10 get Reward: 5
i = 14|Start at 26 results at 56 get Reward: 20


In [4]:
class MAB_agent:
  def __init__(self, envir, init_location):
    # Trace the reward
    self.reward_trace = []
    # initialize the first location
    self.location_now = init_location
    # TODO: implement other features to the agent so it can perform MAB algorithm
    self.lastAction = None
    self.lastState = None
    self.value_table = {}    # format: {state : {action : [value, count]}}

  def get_TotalReward(self):
    return np.sum(self.reward_trace)

  # Running in Simulator
  def getAction(self, observation): 
    self.location_now, action_space, pre_reward = observation
    # NOTICE: the first observation is (init_location, [0,1,2,3], None)
    # You should process the 'None' 

    if self.location_now not in self.value_table.keys():
      # if the state has not been observed before, add to the table with [value, count] = [0, 1]
      self.value_table[self.location_now] = {i: [0, 1] for i in action_space}

    if pre_reward is None:
      action = np.random.choice(action_space, p=[1/(len(action_space)) for action in action_space])
    else:
      self.reward_trace.append(pre_reward)
      # Updating with incremental average over reward samples
      value = self.value_table[self.lastState][self.lastAction][0]
      count = self.value_table[self.lastState][self.lastAction][1]

      count += 1
      value += (1/count) * (pre_reward - value)

      self.value_table[self.lastState][self.lastAction][0] = value
      self.value_table[self.lastState][self.lastAction][1] = count

      # get action
      state_dict = self.value_table[self.lastState].values()
      state_dict_array = np.array(list(state_dict))
      value_column = state_dict_array[:,0]
      action = np.argmax(value_column)

    self.lastState = self.location_now
    self.lastAction = action
    # Assert valid action
    assert action in action_space, "INVALID action taken"
    return action

In [5]:
# create your agent, with environment is the pre-declared Envir and init_location set to 0
init_location = 0
dummyAgent = MAB_agent(envir=Envir, init_location=init_location)

num_iter = 1000
log_freq = 10
Data_plot1 = []
Action_record = []

for i in range(num_iter):
  env_observation = (init_location, Envir.action_space, None)
  if i > 0:
    env_observation = Envir.get_Observation(location=dummyAgent.location_now, action=chosen_action)

  chosen_action = dummyAgent.getAction(observation=env_observation)
  Action_record.append(chosen_action)
  if (i + 1) % log_freq == 0:
    aver = np.mean(dummyAgent.reward_trace)
    Data_plot1.append(aver)
    print('iter: ' + str(i + 1) + '\t Total reward: ' + str(dummyAgent.get_TotalReward()) + '\t Average: ' + str(aver))

iter: 10	 Total reward: 0	 Average: 0.0
iter: 20	 Total reward: 0	 Average: 0.0
iter: 30	 Total reward: 0	 Average: 0.0
iter: 40	 Total reward: 0	 Average: 0.0
iter: 50	 Total reward: 0	 Average: 0.0
iter: 60	 Total reward: 0	 Average: 0.0
iter: 70	 Total reward: 0	 Average: 0.0
iter: 80	 Total reward: 0	 Average: 0.0
iter: 90	 Total reward: 0	 Average: 0.0
iter: 100	 Total reward: 0	 Average: 0.0
iter: 110	 Total reward: 0	 Average: 0.0
iter: 120	 Total reward: 0	 Average: 0.0
iter: 130	 Total reward: 0	 Average: 0.0
iter: 140	 Total reward: 0	 Average: 0.0
iter: 150	 Total reward: 0	 Average: 0.0
iter: 160	 Total reward: 0	 Average: 0.0
iter: 170	 Total reward: 0	 Average: 0.0
iter: 180	 Total reward: 0	 Average: 0.0
iter: 190	 Total reward: 0	 Average: 0.0
iter: 200	 Total reward: 0	 Average: 0.0
iter: 210	 Total reward: 0	 Average: 0.0
iter: 220	 Total reward: 0	 Average: 0.0
iter: 230	 Total reward: 0	 Average: 0.0
iter: 240	 Total reward: 0	 Average: 0.0
iter: 250	 Total reward: 

In [6]:
class MABe_agent(MAB_agent):
  def __init__(self, envir, init_location, epsilon):
    super(MABe_agent, self).__init__(envir, init_location)
    self.epsilon = epsilon

  # Override
  def getAction(self, observation): 
    self.location_now, action_space, pre_reward = observation
    # NOTICE: the first observation is (init_location, [0,1,2,3], None)
    # You should process the 'None' 
    if self.location_now not in self.value_table.keys():
      # if the state has not been observed before, add to the table with [value, count] = [0, 1]
      self.value_table[self.location_now] = {i: [0, 1] for i in action_space}
    
    toss = np.random.rand()

    if pre_reward is None or toss < self.epsilon:
      action = np.random.choice(action_space, p=[1/(len(action_space)) for action in action_space])
    else:
      self.reward_trace.append(pre_reward)
      # Updating with incremental average over reward samples
      value = self.value_table[self.lastState][self.lastAction][0]
      count = self.value_table[self.lastState][self.lastAction][1]

      count += 1
      value += (1/count) * (pre_reward - value)

      self.value_table[self.lastState][self.lastAction][0] = value
      self.value_table[self.lastState][self.lastAction][1] = count
      
      # get action
      state_dict = self.value_table[self.lastState].values()
      state_dict_array = np.array(list(state_dict))
      value_column = state_dict_array[:,0]
      action = np.argmax(value_column)

    self.lastState = self.location_now
    self.lastAction = action
    # Assert valid action
    assert action in action_space, "INVALID action taken"
    return action

In [40]:
# Run your MABe agent
# create your agent, with environment is the pre-declared Envir and init_location set to 0
init_location = 0
epsilon=0.5
dummyAgent = MABe_agent(envir=Envir, init_location=init_location, epsilon=epsilon)

num_iter = 10000
log_freq = 100
Data_plot2 = []
Action_record = []
Location_record = []

for i in range(num_iter):
  env_observation = (init_location, Envir.action_space, None)
  if i > 0:
    env_observation = Envir.get_Observation(location=dummyAgent.location_now, action=chosen_action)

  chosen_action = dummyAgent.getAction(observation=env_observation)
  Location_record.append(env_observation[0])
  Action_record.append(chosen_action)

  if (i + 1) % log_freq == 0:
    aver = np.mean(dummyAgent.reward_trace)
    Data_plot2.append(aver)
    print('iter: ' + str(i + 1) + '\t Total reward: ' + str(dummyAgent.get_TotalReward()) + '\t Average: ' + str(aver))

iter: 100	 Total reward: -35	 Average: -0.813953488372093
iter: 200	 Total reward: -65	 Average: -0.6989247311827957
iter: 300	 Total reward: -90	 Average: -0.6382978723404256
iter: 400	 Total reward: -100	 Average: -0.5208333333333334
iter: 500	 Total reward: -90	 Average: -0.35856573705179284
iter: 600	 Total reward: -100	 Average: -0.33112582781456956
iter: 700	 Total reward: -130	 Average: -0.37790697674418605
iter: 800	 Total reward: -150	 Average: -0.3768844221105528
iter: 900	 Total reward: -235	 Average: -0.5210643015521065
iter: 1000	 Total reward: -245	 Average: -0.4841897233201581
iter: 1100	 Total reward: -345	 Average: -0.6117021276595744
iter: 1200	 Total reward: -390	 Average: -0.6382978723404256
iter: 1300	 Total reward: -370	 Average: -0.5589123867069486
iter: 1400	 Total reward: -370	 Average: -0.5138888888888888
iter: 1500	 Total reward: -470	 Average: -0.6041131105398457
iter: 1600	 Total reward: -525	 Average: -0.6325301204819277
iter: 1700	 Total reward: -515	 Ave

Estimation updates:
>Q-learning:
$Q(S_t,A_t) = (1-\alpha)Q(S_t,A_t) + \alpha(R + \gamma max_a Q(S_{t+1},a))$

In [42]:
# Implement your agent
class Q_agent(MABe_agent):
  def __init__(self, envir, init_location, epsilon):
    super(Q_agent, self).__init__(envir, init_location, epsilon)
    # TODO: initialize your Q table
    self.Q_table = {}
    self.alpha = 0.7
    self.gamma = 0.9  # discount factor

  # Overide method
  def getAction(self, observation):
    # TODO: return your action
    location_now, action_space, pre_reward = observation
    # NOTICE: the first observation is (NONE, [0,1,2,3], None)
    # You should process the 'None' value
    if location_now is not None:
      self.location_now = location_now

    if pre_reward is not None:
      self.reward_trace.append(pre_reward)
      
    # example: get random action

    toss = np.random.rand()

    if pre_reward is None or toss < self.epsilon:
      action = np.random.choice(action_space, p=[1/(len(action_space)) for action in action_space])
    else:
      # implement Q-learning update
      if (self.lastState, self.lastAction) not in self.Q_table.keys():
        self.Q_table[(self.lastState, self.lastAction)] = 0
      # Q-learning update: Q(S,A) += α[R + γ max Q(S',a') - Q(S,A)]
      old_value = self.Q_table[(self.lastState, self.lastAction)]
      state_action_values = [self.Q_table.get((self.location_now, a), 0) for a in action_space]
      max_next_q = np.max(state_action_values)
      new_value = old_value + self.alpha * (pre_reward + self.gamma * max_next_q - old_value)
      self.Q_table[(self.lastState, self.lastAction)] = new_value
      # get action
      action = np.argmax(state_action_values)

    self.lastState = self.location_now
    self.lastAction = action

    # action = np.random.choice(action_space, p=[1 / len(action_space) for _ in action_space])

    # Assert valid action
    assert action in action_space, "INVALID action taken"
    return action

## SARSA

SARSA is the on-policy version of Q-learning
The different is observable through its updating rule

> $Q(S_t, A_t) = (1-\alpha)Q(S_t,A_t) + \alpha(R + \gamma Q(S_{t+1},A_{t+1}))$

Meaning: the sequence of observation is
> $S_t \to A_t \to R \to S_{t+1} \to A_{t+1}$ 

*HENCE*: **S-A-R-S-A**


In [43]:
class SARSA_agent(Q_agent):
  def __init__(self, envir, init_location, epsilon):
    super(SARSA_agent, self).__init__(envir, init_location, epsilon)
  
  # Override method getAction() to implement SARSA algorithm
  def getAction(self, observation):
    location_now, action_space, pre_reward = observation
    # NOTICE: the first observation is (NONE, [0,1,2,3], None)
    # You should process the 'None' value
    if location_now is not None:
      self.location_now = location_now

    if pre_reward is not None:
      self.reward_trace.append(pre_reward)

    toss = np.random.rand()
    if pre_reward is None or toss < self.epsilon:
      action = np.random.choice(action_space, p=[1/(len(action_space)) for action in action_space])
    else:
      if (self.lastState, self.lastAction) not in self.Q_table.keys():
        self.Q_table[(self.lastState, self.lastAction)] = 0
      old_value = self.Q_table[(self.lastState, self.lastAction)]
      # Q(s,a) += α[R + γ Q(s + 1, a + 1) - Q(s, a)]
      next_action = np.random.choice(action_space, p=[1/(len(action_space)) for action in action_space]) if np.random.rand() < self.epsilon else np.argmax([self.Q_table.get((self.location_now, a), 0) for a in action_space])
      next_q = self.Q_table.get((self.location_now, next_action), 0)
      new_value = old_value + self.alpha * (pre_reward + self.gamma * next_q - old_value)
      self.Q_table[(self.lastState, self.lastAction)] = new_value
      action = next_action
      
    self.lastState = self.location_now
    self.lastAction = action

    # Assert valid action
    assert action in action_space, "INVALID action taken"
    return action

In [44]:
# Run your Qlearning agent
# create your agent, with environment is the pre-declared Envir and init_location set to 0
init_location = 0
epsilon=0.5
dummyAgent = Q_agent(envir=Envir, init_location=init_location, epsilon=epsilon)

num_iter = 10000
log_freq = 100
Data_plot2 = []
Action_record = []
Location_record = []

for i in range(num_iter):
  env_observation = (init_location, Envir.action_space, None)
  if i > 0:
    env_observation = Envir.get_Observation(location=dummyAgent.location_now, action=chosen_action)

  chosen_action = dummyAgent.getAction(observation=env_observation)
  Location_record.append(env_observation[0])
  Action_record.append(chosen_action)

  if (i + 1) % log_freq == 0:
    aver = np.mean(dummyAgent.reward_trace)
    Data_plot2.append(aver)
    print('iter: ' + str(i + 1) + '\t Total reward: ' + str(dummyAgent.get_TotalReward()) + '\t Average: ' + str(aver))

iter: 100	 Total reward: -30	 Average: -0.30303030303030304
iter: 200	 Total reward: -150	 Average: -0.7537688442211056
iter: 300	 Total reward: -95	 Average: -0.3177257525083612
iter: 400	 Total reward: -160	 Average: -0.40100250626566414
iter: 500	 Total reward: -265	 Average: -0.531062124248497
iter: 600	 Total reward: -345	 Average: -0.5759599332220368
iter: 700	 Total reward: -390	 Average: -0.5579399141630901
iter: 800	 Total reward: -480	 Average: -0.6007509386733417
iter: 900	 Total reward: -580	 Average: -0.6451612903225806
iter: 1000	 Total reward: -690	 Average: -0.6906906906906907
iter: 1100	 Total reward: -745	 Average: -0.6778889899909009
iter: 1200	 Total reward: -790	 Average: -0.658882402001668
iter: 1300	 Total reward: -850	 Average: -0.6543494996150885
iter: 1400	 Total reward: -960	 Average: -0.686204431736955
iter: 1500	 Total reward: -1040	 Average: -0.6937958639092728
iter: 1600	 Total reward: -1055	 Average: -0.6597873671044403
iter: 1700	 Total reward: -1090	 A

In [45]:
# Run your SARSA agent
# create your agent, with environment is the pre-declared Envir and init_location set to 0
init_location = 0
epsilon=0.5
dummyAgent = SARSA_agent(envir=Envir, init_location=init_location, epsilon=epsilon)

num_iter = 10000
log_freq = 100
Data_plot2 = []
Action_record = []
Location_record = []

for i in range(num_iter):
  env_observation = (init_location, Envir.action_space, None)
  if i > 0:
    env_observation = Envir.get_Observation(location=dummyAgent.location_now, action=chosen_action)

  chosen_action = dummyAgent.getAction(observation=env_observation)
  Location_record.append(env_observation[0])
  Action_record.append(chosen_action)

  if (i + 1) % log_freq == 0:
    aver = np.mean(dummyAgent.reward_trace)
    Data_plot2.append(aver)
    print('iter: ' + str(i + 1) + '\t Total reward: ' + str(dummyAgent.get_TotalReward()) + '\t Average: ' + str(aver))

iter: 100	 Total reward: -100	 Average: -1.0101010101010102
iter: 200	 Total reward: -160	 Average: -0.8040201005025126
iter: 300	 Total reward: -305	 Average: -1.020066889632107
iter: 400	 Total reward: -425	 Average: -1.0651629072681705
iter: 500	 Total reward: -485	 Average: -0.9719438877755511
iter: 600	 Total reward: -545	 Average: -0.9098497495826378
iter: 700	 Total reward: -635	 Average: -0.9084406294706724
iter: 800	 Total reward: -755	 Average: -0.9449311639549437
iter: 900	 Total reward: -830	 Average: -0.9232480533926585
iter: 1000	 Total reward: -950	 Average: -0.950950950950951
iter: 1100	 Total reward: -1025	 Average: -0.9326660600545951
iter: 1200	 Total reward: -1055	 Average: -0.8798999165971643
iter: 1300	 Total reward: -1115	 Average: -0.8583525789068515
iter: 1400	 Total reward: -1190	 Average: -0.8506075768406004
iter: 1500	 Total reward: -1310	 Average: -0.8739159439626417
iter: 1600	 Total reward: -1445	 Average: -0.9036898061288305
iter: 1700	 Total reward: -15